In [1]:
import re
import pandas as pd
import numpy as np
import time
import random
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    BaggingClassifier, VotingClassifier, StackingClassifier,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import mlflow



SEED     = 2007
CV_FOLDS = 3
np.random.seed(SEED)
random.seed(SEED)

BASE_DIR = Path('d:\\mlops-experiments\\experiments\\ensemble_pyramid.py').resolve().parent


print("=" * 65)
print("🏗️  ENSEMBLE PYRAMID — 6 Camadas de Ensembles sobre Ensembles")
print("=" * 65)
mlflow.set_experiment("Ensemble_Pyramid")
mlflow.start_run(run_name="ensemble_pyramid")


# ─── Funções de Limpeza ────────────────────────────────────────────────────────
def clean_tweet(text: str) -> str:
    """Limpeza básica de tweet para análise de sentimento."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # URLs
    text = re.sub(r'@\w+', '', text)                     # Menções
    text = re.sub(r'#(\w+)', r'\1', text)               # Hashtags → palavra
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', '', text)    # Caracteres especiais
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ─── 1. Dados ──────────────────────────────────────────────────────────────────
print("\n📂 Carregando dados...")
cols = ['tweet_id', 'entity', 'sentiment', 'text']
train_path = BASE_DIR / "nlp" / "twitter-entity-sentiment" / "senti-pred-variations" / "senti-pred-exp1" / "data" / "raw" / "twitter_training.csv"
val_path = BASE_DIR / "nlp" / "twitter-entity-sentiment" / "senti-pred-variations" / "senti-pred-exp1" / "data" / "raw" / "twitter_validation.csv"
train_df = pd.read_csv(train_path, names=cols, header=None)
val_df   = pd.read_csv(val_path, names=cols, header=None)

# Aplica limpeza
for df in (train_df, val_df):
    df['clean_text'] = df['text'].apply(clean_tweet)

# Remove registros sem texto e sem sentimento válido
valid_sentiments = ['Positive', 'Negative', 'Neutral', 'Irrelevant']
train_df = train_df[
    (train_df['clean_text'].str.len() > 0) &
    (train_df['sentiment'].isin(valid_sentiments))
].copy()

val_df = val_df[
    (val_df['clean_text'].str.len() > 0) &
    (val_df['sentiment'].isin(valid_sentiments))
].copy()

TEXT_COL   = "clean_text"
TARGET_COL = "sentiment"

le = LabelEncoder()
le.fit(train_df[TARGET_COL])
y_train = le.transform(train_df[TARGET_COL])
y_val   = le.transform(val_df[TARGET_COL])
CLASSES = list(le.classes_)

print(f"Treino    : {train_df.shape[0]:,} amostras")
print(f"Validação : {val_df.shape[0]:,} amostras")
print(f"Classes   : {CLASSES}")
print(f"Treino após limpeza : {train_df.shape}")
print(f"Validação após limpeza: {val_df.shape}")
print(train_df[['text', 'clean_text', 'sentiment']].head(3))

# ─── 2. TF-IDF — mantém SPARSE ────────────────────────────────────────────────
print("\n🔤 Gerando features TF-IDF (sparse)...")
tfidf = TfidfVectorizer(
    max_features  = 70_000,
    ngram_range   = (1, 2),
    sublinear_tf  = True,
    min_df        = 2,
    strip_accents = "unicode",
)
X_train = tfidf.fit_transform(train_df[TEXT_COL].fillna(""))
X_val   = tfidf.transform(val_df[TEXT_COL].fillna(""))

print(f"Shape    : {X_train.shape}")
print(f"Formato  : {type(X_train).__name__} (sparse ✔)")
print(f"RAM ~    : {X_train.data.nbytes / 1e6:.1f} MB")

# ─── Utilitário de avaliação ───────────────────────────────────────────────────
results = []

def evaluate(name, model, X, y, layer):
    preds = model.predict(X)
    acc   = accuracy_score(y, preds)
    f1    = f1_score(y, preds, average="weighted")
    bar   = "█" * int(f1 * 35)
    print(f"    ✔ F1={f1:.4f}  Acc={acc:.4f}  {bar}")
    results.append({"layer": layer, "name": name, "acc": acc, "f1": f1})

class PreFittedSoftVoting:
    def __init__(self, estimators):
        self.estimators = estimators
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        probs = [est.predict_proba(X) for est in self.estimators]
        return np.mean(probs, axis=0)
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

class PreFittedHardVoting:
    def __init__(self, estimators):
        self.estimators = estimators
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        P = np.column_stack([est.predict(X) for est in self.estimators])
        n_classes = len(CLASSES)
        n_estimators = P.shape[1]
        out = np.zeros((P.shape[0], n_classes), dtype=float)
        for i, row in enumerate(P):
            vals, cnts = np.unique(row, return_counts=True)
            out[i, vals] = cnts
        return out / n_estimators
    def predict(self, X):
        P = np.column_stack([est.predict(X) for est in self.estimators])
        out = []
        for row in P:
            vals, cnts = np.unique(row, return_counts=True)
            out.append(vals[np.argmax(cnts)])
        return np.array(out)

class MetaStackingLR:
    def __init__(self, estimators, C=0.5):
        self.estimators = estimators
        self.model = LogisticRegression(C=C, max_iter=1000, random_state=SEED, solver="lbfgs", n_jobs=-1)
    def _Z(self, X):
        return np.hstack([est.predict_proba(X) for est in self.estimators])
    def fit(self, X, y):
        Z = self._Z(X)
        self.model.fit(Z, y)
        return self
    def predict_proba(self, X):
        Z = self._Z(X)
        return self.model.predict_proba(Z)
    def predict(self, X):
        Z = self._Z(X)
        return self.model.predict(Z)

# ─── Fábricas ─────────────────────────────────────────────────────────────────
def make_lr(C=11.0):
    return LogisticRegression(C=C, max_iter=1000, solver="lbfgs",
                               multi_class="auto", random_state=SEED, n_jobs=-1)

# ⚙️ Para voting="hard" e contextos que NÃO precisam de predict_proba:
#    LinearSVC puro — idêntico ao notebook, sem overhead de calibração
def make_svc_hard():
    return LinearSVC(C=19.0, max_iter=1000, random_state=SEED)

# ⚙️ Para voting="soft", Stacking e Bagging que precisam de predict_proba:
#    CalibratedClassifierCV envolve o LinearSVC e aprende uma sigmoid por cima
def make_svc_soft():
    return CalibratedClassifierCV(
        LinearSVC(C=19.0, max_iter=1000, random_state=SEED),
        cv=2, method="sigmoid"
    )

def make_ridge_hard():
    return RidgeClassifier()

def make_ridge_soft():
    return CalibratedClassifierCV(RidgeClassifier(alpha=1.0), cv=2, method="sigmoid")

def make_nb():
    return MultinomialNB(alpha=0.1)

def make_cnb():
    return ComplementNB(alpha=0.1)

def make_rf(n=100):
    return RandomForestClassifier(n_estimators=n, random_state=SEED, n_jobs=-1)

def make_et(n=100):
    return ExtraTreesClassifier(n_estimators=n, random_state=SEED, n_jobs=-1)

# ═══════════════════════════════════════════════════════════════════════════════
# CAMADA 1 — BASE LEARNERS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 65)
print("🧱 CAMADA 1 — Base Learners")
print("═" * 65)

t0 = time.time()

base_learners = {
    "LogisticRegression" : make_lr(),
    "LinearSVC"          : make_svc_hard(),   # puro, sem calibração
    "MultinomialNB"      : make_nb(),
    "ComplementNB"       : make_cnb(),
    "Ridge"              : make_ridge_hard(),  # puro
    "RandomForest"       : make_rf(100),
    "ExtraTrees"         : make_et(100),
}

trained_base = {}
for name, model in base_learners.items():
    print(f"\n  [{name}]")
    t = time.time()
    model.fit(X_train, y_train)
    trained_base[name] = model
    evaluate(name, model, X_val, y_val, layer=1)
    print(f"    ⏱  {time.time()-t:.1f}s")

print(f"\n⏱️  Camada 1 total: {time.time()-t0:.1f}s")

# ═══════════════════════════════════════════════════════════════════════════════
# CAMADA 2 — ENSEMBLES DOS BASE LEARNERS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 65)
print("⚙️  CAMADA 2 — Ensembles dos Base Learners")
print("═" * 65)

# C2A — Bagging(LR) + Bagging(SVC_soft) → Voting Soft
# Bagging precisa de predict_proba para voting soft → make_svc_soft()
# Reduzido para 5 estimators para acelerar
print("\n  [C2A] Bagging(LR) + Bagging(SVC) → Voting Soft")
c2a_bag_lr  = BaggingClassifier(estimator=make_lr(),       n_estimators=5,
                                  max_samples=0.8, random_state=SEED,   n_jobs=-1)
c2a_bag_svc = BaggingClassifier(estimator=make_svc_soft(), n_estimators=5,
                                  max_samples=0.8, random_state=SEED+1, n_jobs=-1)
c2a_bag_lr.fit(X_train, y_train)
c2a_bag_svc.fit(X_train, y_train)
c2a = VotingClassifier(
    estimators=[("bag_lr", c2a_bag_lr), ("bag_svc", c2a_bag_svc)],
    voting="soft", n_jobs=-1
)
c2a.fit(X_train, y_train)
evaluate("C2A: Bagging(LR+SVC)->Voting Soft", c2a, X_val, y_val, layer=2)

# C2B — Voting Soft (NB + CNB + Ridge_soft)
# Ridge não tem predict_proba → make_ridge_soft()
print("\n  [C2B] Voting Soft (NB + ComplementNB + Ridge)")
c2b = VotingClassifier(
    estimators=[("nb", make_nb()), ("cnb", make_cnb()), ("rdg", make_ridge_soft())],
    voting="soft", n_jobs=-1
)
c2b.fit(X_train, y_train)
evaluate("C2B: Voting Soft(NB+CNB+Ridge)", c2b, X_val, y_val, layer=2)

# C2C — Stacking (RF + ET → LR meta)
# Stacking usa predict_proba internamente → RF e ET já têm nativamente
# Reduzido para 50 estimators para acelerar
print("\n  [C2C] Stacking (RF + ET → LR meta)")
c2c = StackingClassifier(
    estimators=[("rf", make_rf(30)), ("et", make_et(30))],
    final_estimator=make_lr(),
    cv=CV_FOLDS, passthrough=False, n_jobs=-1
)
c2c.fit(X_train, y_train)
evaluate("C2C: Stacking(RF+ET->LR)", c2c, X_val, y_val, layer=2)

# C2D — Voting Hard (SVC_hard + LR + Ridge_hard)
# Hard voting usa apenas predict() → LinearSVC e Ridge puros
print("\n  [C2D] Voting Hard (SVC + LR + Ridge)")
c2d = VotingClassifier(
    estimators=[("svc", make_svc_hard()), ("lr", make_lr()), ("rdg", make_ridge_hard())],
    voting="hard", n_jobs=-1
)
c2d.fit(X_train, y_train)
evaluate("C2D: Voting Hard(SVC+LR+Ridge)", c2d, X_val, y_val, layer=2)

# C2E — Bagging(NB) + Bagging(CNB) → Voting Soft
# NB e CNB já têm predict_proba nativamente
# Reduzido para 8 estimators para acelerar
print("\n  [C2E] Bagging(NB) + Bagging(CNB) → Voting Soft")
c2e_bag_nb  = BaggingClassifier(estimator=make_nb(),  n_estimators=6,
                                  max_samples=0.8, random_state=SEED,   n_jobs=-1)
c2e_bag_cnb = BaggingClassifier(estimator=make_cnb(), n_estimators=6,
                                  max_samples=0.8, random_state=SEED+2, n_jobs=-1)
c2e_bag_nb.fit(X_train, y_train)
c2e_bag_cnb.fit(X_train, y_train)
c2e = VotingClassifier(
    estimators=[("bag_nb", c2e_bag_nb), ("bag_cnb", c2e_bag_cnb)],
    voting="soft", n_jobs=-1
)
c2e.fit(X_train, y_train)
evaluate("C2E: Bagging(NB+CNB)->Voting Soft", c2e, X_val, y_val, layer=2)

print("\n⏱️  Camada 2 concluída.")

# ═══════════════════════════════════════════════════════════════════════════════
# CAMADA 3 — ENSEMBLES DOS ENSEMBLES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 65)
print("🔗 CAMADA 3 — Ensembles dos Ensembles")
print("═" * 65)

# C3A — Stacking (C2A + C2B → LR)
# C2A e C2B são VotingClassifier soft → já têm predict_proba
print("\n  [C3A] Stacking (C2A + C2B → LR)")
c3a = StackingClassifier(
    estimators=[("c2a", c2a), ("c2b", c2b)],
    final_estimator=make_lr(),
    cv=CV_FOLDS, passthrough=True, n_jobs=-1
)
c3a.fit(X_train, y_train)
evaluate("C3A: Stacking(C2A+C2B->LR)", c3a, X_val, y_val, layer=3)

# C3B — Bagging sobre Stacking clonável (RF+ET→LR)
# RF e ET têm predict_proba → ok para Stacking interno
# Reduzido para 30 estimators e 3 bags para acelerar
print("\n  [C3B] Bagging sobre Stacking (RF+ET→LR)")
c3b = BaggingClassifier(
    estimator=StackingClassifier(
        estimators=[("rf", make_rf(20)), ("et", make_et(20))],
        final_estimator=make_lr(),
        cv=CV_FOLDS, n_jobs=-1
    ),
    n_estimators=2,
    max_samples=0.8,
    random_state=SEED,
    n_jobs=-1,
)
c3b.fit(X_train, y_train)
evaluate("C3B: Bagging(Stacking)", c3b, X_val, y_val, layer=3)

print("\n  [C3C] Voting Soft (C2B + C2E)")
c3c = VotingClassifier(
    estimators=[("c2b", c2b), ("c2e", c2e)],
    voting="soft", n_jobs=-1
)
c3c.fit(X_train, y_train)
evaluate("C3C: Voting Soft(C2B+C2E)", c3c, X_val, y_val, layer=3)

print("\n⏱️  Camada 3 concluída.")

# ═══════════════════════════════════════════════════════════════════════════════
# CAMADA 4 — META-ENSEMBLE FINAL
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 65)
print("🏆 CAMADA 4 — Meta-Ensemble Final")
print("═" * 65)

print("\n  [C4A] Meta Voting Soft (C3A + C3B + C3C)")
c4a = PreFittedSoftVoting([c3a, c3b, c3c]).fit(X_train, y_train)
evaluate("C4A: Meta Voting Soft(C3A+C3B+C3C)", c4a, X_val, y_val, layer=4)

print("\n  [C4B] Meta Stacking (C3A + C3B + C3C → LR)")
c4b = MetaStackingLR([c3a, c3b, c3c], C=0.5).fit(X_train, y_train)
evaluate("C4B: Meta Stacking(->LR)", c4b, X_val, y_val, layer=4)

print("\n  [C4C] Meta Voting Hard (C3A + C3B + C3C + C2C)")
c4c = PreFittedHardVoting([c3a, c3b, c3c, c2c]).fit(X_train, y_train)
evaluate("C4C: Meta Voting Hard(C3s+C2C)", c4c, X_val, y_val, layer=4)

print("\n⏱️  Camada 4 concluída.")

# ═══════════════════════════════════════════════════════════════════════════════
# RESULTADOS FINAIS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 65)
print("� CAMADA 5 — Meta-Ensemble Intermediário")
print("═" * 65)

print("\n  [C5A] Meta2 Voting Soft (C4A + C4B + C4C)")
c5a = PreFittedSoftVoting([c4a, c4b, c4c]).fit(X_train, y_train)
evaluate("C5A: Meta2 Voting Soft(C4A+C4B+C4C)", c5a, X_val, y_val, layer=5)

print("\n  [C5B] Meta2 Stacking (C4A + C4B + C4C → LR)")
c5b = MetaStackingLR([c4a, c4b, c4c], C=0.4).fit(X_train, y_train)
evaluate("C5B: Meta2 Stacking(->LR)", c5b, X_val, y_val, layer=5)

print("\n  [C5C] Meta2 Voting Hard (C4A + C4B + C4C)")
c5c = PreFittedHardVoting([c4a, c4b, c4c]).fit(X_train, y_train)
evaluate("C5C: Meta2 Voting Hard(C4s)", c5c, X_val, y_val, layer=5)

print("\n" + "═" * 65)
print("💎 CAMADA 6 — Meta-Ensemble Final Aprimorado")
print("═" * 65)

print("\n  [C6A] Final Stacking (C5A + C5B + C5C → LR)")
c6a = MetaStackingLR([c5a, c5b, c5c], C=0.3).fit(X_train, y_train)
evaluate("C6A: Final Stacking(->LR)", c6a, X_val, y_val, layer=6)

print("\n  [C6B] Final Voting Soft (C5A + C5B + C5C)")
c6b = PreFittedSoftVoting([c5a, c5b, c5c]).fit(X_train, y_train)
evaluate("C6B: Final Voting Soft(C5s)", c6b, X_val, y_val, layer=6)

print("\n  [C6C] Final Voting Hard (C5A + C5B + C5C + C4B)")
c6c = PreFittedHardVoting([c5a, c5b, c5c, c4b]).fit(X_train, y_train)
evaluate("C6C: Final Voting Hard(C5s+C4B)", c6c, X_val, y_val, layer=6)

print("\n" + "═" * 65)
print("�📊 COMPARAÇÃO COMPLETA — Todas as Camadas")
print("═" * 65)

df_results = pd.DataFrame(results).sort_values(["layer", "f1"], ascending=[True, False])
layer_names = {
    1: "Base Learners",
    2: "Ensembles L1",
    3: "Ensembles de Ensembles",
    4: "Meta-Ensemble Final",
    5: "Meta-Ensemble Intermediário",
    6: "Meta-Ensemble Final Aprimorado",
}

for layer in [1, 2, 3, 4, 5, 6]:
    print(f"\n── Camada {layer}: {layer_names[layer]} ──")
    subset = df_results[df_results["layer"] == layer]
    for _, row in subset.iterrows():
        bar = "█" * int(row["f1"] * 35)
        print(f"  {row['name']:<45} F1={row['f1']:.4f}  {bar}")

best = df_results.loc[df_results["f1"].idxmax()]
print(f"\n🏆 MELHOR MODELO GERAL : {best['name']}")
print(f"   Camada {int(best['layer'])} | Acc={best['acc']:.4f} | F1={best['f1']:.4f}")

# ─── Classification report do melhor ──────────────────────────────────────────
model_map = {
    "C4A: Meta Voting Soft(C3A+C3B+C3C)"     : c4a,
    "C4B: Meta Stacking(->LR)"               : c4b,
    "C4C: Meta Voting Hard(C3s+C2C)"         : c4c,
    "C5A: Meta2 Voting Soft(C4A+C4B+C4C)"    : c5a,
    "C5B: Meta2 Stacking(->LR)"              : c5b,
    "C5C: Meta2 Voting Hard(C4s)"            : c5c,
    "C6A: Final Stacking(->LR)"              : c6a,
    "C6B: Final Voting Soft(C5s)"            : c6b,
    "C6C: Final Voting Hard(C5s+C4B)"        : c6c,
    "C3A: Stacking(C2A+C2B->LR)"         : c3a,
    "C3B: Bagging(Stacking)"              : c3b,
    "C3C: Voting Soft(C2B+C2E)"           : c3c,
    "C2A: Bagging(LR+SVC)->Voting Soft"   : c2a,
    "C2B: Voting Soft(NB+CNB+Ridge)"      : c2b,
    "C2C: Stacking(RF+ET->LR)"            : c2c,
    "C2D: Voting Hard(SVC+LR+Ridge)"      : c2d,
    "C2E: Bagging(NB+CNB)->Voting Soft"   : c2e,
    **trained_base,
}

best_obj = model_map.get(best["name"])
if best_obj:
    print(f"\n=== Classification Report — {best['name']} ===")
    print(classification_report(y_val, best_obj.predict(X_val), target_names=CLASSES))

# ─── Análise de ganho por camada ──────────────────────────────────────────────
print("═" * 65)
print("🔬 ANÁLISE DA PIRÂMIDE — Ganho por Camada")
print("═" * 65)

layer_bests = [df_results[df_results["layer"] == l]["f1"].max() for l in [1, 2, 3, 4, 5, 6]]
layer_means = [df_results[df_results["layer"] == l]["f1"].mean() for l in [1, 2, 3, 4, 5, 6]]

for i, (lb, lm) in enumerate(zip(layer_bests, layer_means), start=1):
    print(f"  Camada {i} → Melhor F1: {lb:.4f} | Média F1: {lm:.4f}")

print(f"\n  Ganho C1→C2 : {(layer_bests[1]-layer_bests[0])*100:+.2f}pp")
print(f"  Ganho C2→C3 : {(layer_bests[2]-layer_bests[1])*100:+.2f}pp")
print(f"  Ganho C3→C4 : {(layer_bests[3]-layer_bests[2])*100:+.2f}pp")
print(f"  Ganho C4→C5 : {(layer_bests[4]-layer_bests[3])*100:+.2f}pp")
print(f"  Ganho C5→C6 : {(layer_bests[5]-layer_bests[4])*100:+.2f}pp")
print(f"  Ganho Total : {(layer_bests[5]-layer_bests[0])*100:+.2f}pp")

# ─── Salvar ───────────────────────────────────────────────────────────────────
results_path = BASE_DIR / "ensemble_pyramid_results.csv"
df_results.to_csv(results_path, index=False)
mlflow.log_artifact(str(results_path))
print(f"\nResultados salvos: {results_path}")

if best_obj:
    model_path = BASE_DIR / "ensemble_pyramid_best.pkl"
    joblib.dump({"model": best_obj, "tfidf": tfidf, "encoder": le}, model_path)
    mlflow.log_artifact(str(model_path))
    mlflow.log_param("artifact_version", "ensemble_pyramid")
    print(f"Melhor modelo salvo: {model_path}")

print("\n✅ Ensemble Pyramid concluído!")
mlflow.end_run()

🏗️  ENSEMBLE PYRAMID — 6 Camadas de Ensembles sobre Ensembles


2026/08/12 19:41:36 INFO mlflow.tracking.fluent: Experiment with name 'Ensemble_Pyramid' does not exist. Creating a new experiment.



📂 Carregando dados...


Treino    : 73,763 amostras
Validação : 1,000 amostras
Classes   : ['Irrelevant', 'Negative', 'Neutral', 'Positive']
Treino após limpeza : (73763, 5)
Validação após limpeza: (1000, 5)
                                                text  \
0  im getting on borderlands and i will murder yo...   
1  I am coming to the borders and I will kill you...   
2  im getting on borderlands and i will kill you ...   

                                          clean_text sentiment  
0  im getting on borderlands and i will murder yo...  Positive  
1  i am coming to the borders and i will kill you...  Positive  
2  im getting on borderlands and i will kill you ...  Positive  

🔤 Gerando features TF-IDF (sparse)...


Shape    : (73763, 70000)
Formato  : csr_matrix (sparse ✔)
RAM ~    : 15.2 MB

═════════════════════════════════════════════════════════════════
🧱 CAMADA 1 — Base Learners
═════════════════════════════════════════════════════════════════

  [LogisticRegression]


...

    ✔ F1=0.9810  Acc=0.9810  ██████████████████████████████████

═════════════════════════════════════════════════════════════════
💎 CAMADA 6 — Meta-Ensemble Final Aprimorado
═════════════════════════════════════════════════════════════════

  [C6A] Final Stacking (C5A + C5B + C5C → LR)


    ✔ F1=0.9820  Acc=0.9820  ██████████████████████████████████

  [C6B] Final Voting Soft (C5A + C5B + C5C)


    ✔ F1=0.9810  Acc=0.9810  ██████████████████████████████████

  [C6C] Final Voting Hard (C5A + C5B + C5C + C4B)


    ✔ F1=0.9810  Acc=0.9810  ██████████████████████████████████

═════════════════════════════════════════════════════════════════
�📊 COMPARAÇÃO COMPLETA — Todas as Camadas
═════════════════════════════════════════════════════════════════

── Camada 1: Base Learners ──
  LinearSVC                                     F1=0.9830  ██████████████████████████████████
  Ridge                                         F1=0.9830  ██████████████████████████████████
  LogisticRegression                            F1=0.9801  ██████████████████████████████████
  ExtraTrees                                    F1=0.9760  ██████████████████████████████████
  RandomForest                                  F1=0.9730  ██████████████████████████████████
  MultinomialNB                                 F1=0.9621  █████████████████████████████████
  ComplementNB                                  F1=0.9531  █████████████████████████████████

── Camada 2: Ensembles L1 ──
  C2D: Voting Hard(SVC+LR+Ridge)            

Melhor modelo salvo: D:\mlops-experiments\experiments\ensemble_pyramid_best.pkl

✅ Ensemble Pyramid concluído!
